RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [4]:
import os 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:

### Step 1: Data Ingestion (Document Structure creation)
###Read all the pdf files from the directory
### Its purpose is to find all PDF files inside a directory (including subdirectories), extract their contents page by page, attach metadata, and return all the extracted documents as a single list.

def process_all_pdfs(pdf_directory):
    """Process all PDF files in the specified directory and its subdirectories."""

    all_documents = []  # List to hold all documents
    pdf_dir = Path(pdf_directory)  # Convert to Path object for easier handling

    ### Find all the pdf files recursively in the directory
    pdf_files = list(pdf_dir.glob("**/*.pdf"))  # Recursively find all PDF files

    for pdf_file in pdf_files:
        print(f"Processing file: {pdf_file}")

        try:
            #load the pdf file using PyPDFLoader
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add metadata to each document
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name  # Add the source file name to metadata
                doc.metadata['file_type'] = 'pdf'  # Add the file type to metadata

            all_documents.extend(documents)  # Add the documents to the main list
            print(f"Successfully processed {len(documents)} pages from {pdf_file.name}")

        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")

    print(f"Total documents processed: {len(all_documents)}")
    return all_documents  # Return the list of all documents

###Process all PDFs in the specified directory
pdf_directory = process_all_pdfs("../data/pdf")


Processing file: ../data/pdf/file-example_PDF_500_kB.pdf
Successfully processed 5 pages from file-example_PDF_500_kB.pdf
Processing file: ../data/pdf/file-sample_150kB.pdf
Successfully processed 4 pages from file-sample_150kB.pdf
Total documents processed: 9


In [9]:
pdf_directory

[Document(metadata={'producer': 'LibreOffice 4.2', 'creator': 'Writer', 'creationdate': '2017-08-16T14:42:28+02:00', 'source': '../data/pdf/file-example_PDF_500_kB.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'file-example_PDF_500_kB.pdf', 'file_type': 'pdf'}, page_content='Lorem ipsum \nLorem ipsum dolor sit amet, consectetur adipiscing \nelit. Nunc ac faucibus odio. \nVestibulum neque massa, scelerisque sit amet ligula eu, congue molestie mi. Praesent ut\nvarius sem. Nullam at porttitor arcu, nec lacinia nisi. Ut ac dolor vitae odio interdum\ncondimentum.  Vivamus  dapibus  sodales  ex,  vitae  malesuada  ipsum  cursus\nconvallis. Maecenas sed egestas nulla, ac condimentum orci.  Mauris diam felis,\nvulputate ac suscipit et, iaculis non est. Curabitur semper arcu ac ligula semper, nec luctus\nnisl blandit. Integer lacinia ante ac libero lobortis imperdiet. Nullam mollis convallis ipsum,\nac accumsan nunc vehicula vitae. Nulla eget justo in felis tristique frin

In [10]:
### Step 2: Chunking
### Chunking is the process of breaking down large documents into smaller, manageable pieces (chunks) for easier processing and retrieval. This is especially useful in applications like RAG (Retrieval-Augmented Generation), where we want to retrieve relevant information from large documents efficiently.
### Text splitting into chunks

def split_documents(documents, chunk_size=10000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Splitting {len(documents)} documents into total chunks : {len(split_docs)}")

    # how example fo a chunk 
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")  # Print first 200 characters of the first chunk
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs



In [11]:
chunks = split_documents(pdf_directory)
chunks

Splitting 9 documents into total chunks : 8

Example chunk:
Content: Lorem ipsum 
Lorem ipsum dolor sit amet, consectetur adipiscing 
elit. Nunc ac faucibus odio. 
Vestibulum neque massa, scelerisque sit amet ligula eu, congue molestie mi. Praesent ut
varius sem. Nulla...
Metadata: {'producer': 'LibreOffice 4.2', 'creator': 'Writer', 'creationdate': '2017-08-16T14:42:28+02:00', 'source': '../data/pdf/file-example_PDF_500_kB.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'file-example_PDF_500_kB.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'LibreOffice 4.2', 'creator': 'Writer', 'creationdate': '2017-08-16T14:42:28+02:00', 'source': '../data/pdf/file-example_PDF_500_kB.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'file-example_PDF_500_kB.pdf', 'file_type': 'pdf'}, page_content='Lorem ipsum \nLorem ipsum dolor sit amet, consectetur adipiscing \nelit. Nunc ac faucibus odio. \nVestibulum neque massa, scelerisque sit amet ligula eu, congue molestie mi. Praesent ut\nvarius sem. Nullam at porttitor arcu, nec lacinia nisi. Ut ac dolor vitae odio interdum\ncondimentum.  Vivamus  dapibus  sodales  ex,  vitae  malesuada  ipsum  cursus\nconvallis. Maecenas sed egestas nulla, ac condimentum orci.  Mauris diam felis,\nvulputate ac suscipit et, iaculis non est. Curabitur semper arcu ac ligula semper, nec luctus\nnisl blandit. Integer lacinia ante ac libero lobortis imperdiet. Nullam mollis convallis ipsum,\nac accumsan nunc vehicula vitae. Nulla eget justo in felis tristique frin